In [2]:
import pandas as pd
import numpy as np
import joblib

# Load raw data (to compute team stats)
results      = pd.read_csv('../data/results.csv')
shootouts    = pd.read_csv('../data/shootouts.csv')
former_names = pd.read_csv('../data/former_names.csv')

results['date']   = pd.to_datetime(results['date'])
shootouts['date'] = pd.to_datetime(shootouts['date'])

name_map = dict(zip(former_names['former'], former_names['current']))
def normalize(name): return name_map.get(name, name)

results['home_team'] = results['home_team'].apply(normalize)
results['away_team'] = results['away_team'].apply(normalize)
shootouts['home_team'] = shootouts['home_team'].apply(normalize)
shootouts['away_team'] = shootouts['away_team'].apply(normalize)
shootouts['winner']    = shootouts['winner'].apply(normalize)

results_sorted = results.dropna(subset=['home_score','away_score']).sort_values('date').reset_index(drop=True)

In [3]:
def team_stats(team, before_date, n=20):
    past = results_sorted[
        (results_sorted['date'] < before_date) &
        ((results_sorted['home_team'] == team) | (results_sorted['away_team'] == team))
    ].tail(n)
    if len(past) == 0:
        return {'win_rate': 0.5, 'avg_scored': 1.0, 'avg_conceded': 1.0, 'n_matches': 0}
    wins, scored, conceded = [], [], []
    for _, m in past.iterrows():
        if m['home_team'] == team:
            wins.append(1 if m['home_score'] > m['away_score'] else 0)
            scored.append(m['home_score']); conceded.append(m['away_score'])
        else:
            wins.append(1 if m['away_score'] > m['home_score'] else 0)
            scored.append(m['away_score']); conceded.append(m['home_score'])
    return {'win_rate': np.mean(wins), 'avg_scored': np.mean(scored),
            'avg_conceded': np.mean(conceded), 'n_matches': len(past)}

def shootout_win_rate(team, before_date):
    past = shootouts[
        (shootouts['date'] < before_date) &
        ((shootouts['home_team'] == team) | (shootouts['away_team'] == team))
    ]
    if len(past) == 0: return 0.5
    return (past['winner'] == team).sum() / len(past)

In [4]:
# Load models
model_outcome    = joblib.load('../dataII/model_outcome.pkl')
model_home_score = joblib.load('../dataII/model_home_score.pkl')
model_away_score = joblib.load('../dataII/model_away_score.pkl')

FEATURES = [
    'is_neutral', 'year', 'month',
    'home_win_rate', 'home_avg_scored', 'home_avg_conceded', 'home_n_matches', 'home_shootout_wr',
    'away_win_rate', 'away_avg_scored', 'away_avg_conceded', 'away_n_matches', 'away_shootout_wr',
]

def predict(home_team, away_team, is_neutral=1):
    today = pd.Timestamp.today()
    h = team_stats(home_team, today)
    a = team_stats(away_team, today)

    row = pd.DataFrame([{
        'is_neutral': is_neutral, 'year': today.year, 'month': today.month,
        'home_win_rate': h['win_rate'], 'home_avg_scored': h['avg_scored'],
        'home_avg_conceded': h['avg_conceded'], 'home_n_matches': h['n_matches'],
        'home_shootout_wr': shootout_win_rate(home_team, today),
        'away_win_rate': a['win_rate'], 'away_avg_scored': a['avg_scored'],
        'away_avg_conceded': a['avg_conceded'], 'away_n_matches': a['n_matches'],
        'away_shootout_wr': shootout_win_rate(away_team, today),
    }])

    outcome = model_outcome.predict(row)[0]
    proba   = model_outcome.predict_proba(row)[0]
    home_g  = max(0, int(round(model_home_score.predict(row)[0])))
    away_g  = max(0, int(round(model_away_score.predict(row)[0])))
    label   = {0: 'Home Win', 1: 'Draw', 2: 'Away Win'}

    print(f'{home_team} vs {away_team}')
    print(f'Predicted score : {home_team} {home_g} - {away_g} {away_team}')
    print(f'Outcome         : {label[outcome]}')
    print(f'Probabilities   : Home {proba[0]*100:.1f}%  Draw {proba[1]*100:.1f}%  Away {proba[2]*100:.1f}%')


In [5]:
predict('Argentina', 'Austria')
predict('France', 'Iraq')

Argentina vs Austria
Predicted score : Argentina 2 - 1 Austria
Outcome         : Home Win
Probabilities   : Home 68.1%  Draw 21.5%  Away 10.4%
France vs Iraq
Predicted score : France 2 - 1 Iraq
Outcome         : Home Win
Probabilities   : Home 64.2%  Draw 24.7%  Away 11.1%


In [6]:
predict('Norway', 'Senegal')
predict('Jordan', 'Algeria')

Norway vs Senegal
Predicted score : Norway 2 - 1 Senegal
Outcome         : Home Win
Probabilities   : Home 49.0%  Draw 27.4%  Away 23.6%
Jordan vs Algeria
Predicted score : Jordan 1 - 2 Algeria
Outcome         : Away Win
Probabilities   : Home 15.3%  Draw 19.1%  Away 65.6%


In [7]:
predict('Portugal', 'Uzbekistan', is_neutral=1)
predict('England', 'Ghana', is_neutral=1)

Portugal vs Uzbekistan
Predicted score : Portugal 2 - 1 Uzbekistan
Outcome         : Home Win
Probabilities   : Home 61.5%  Draw 21.7%  Away 16.8%
England vs Ghana
Predicted score : England 2 - 1 Ghana
Outcome         : Home Win
Probabilities   : Home 64.3%  Draw 28.2%  Away 7.5%


In [8]:
predict('Panama', 'Croatia', is_neutral=1)
predict('Colombia', 'DR Congo', is_neutral=1)

Panama vs Croatia
Predicted score : Panama 1 - 1 Croatia
Outcome         : Home Win
Probabilities   : Home 45.1%  Draw 27.0%  Away 27.9%
Colombia vs DR Congo
Predicted score : Colombia 1 - 1 DR Congo
Outcome         : Home Win
Probabilities   : Home 44.1%  Draw 24.7%  Away 31.2%
